# Day 4 — Python OOP

This notebook covers Python OOP (Classes, `@property`, Composition, Dataclasses, Custom Exceptions)

## Validation goals
- Practice creating classes, methods, and instance initializers (`__init__`)
- Use `@property` getters and setters for encapsulation and data validation
- Define and raise custom domain exceptions (`OutOfStockError`)
- Simplify state-holding classes using the `@dataclass` decorator
- Build an integrated Product / Inventory domain model capstone

**Expected result:** every validation cell runs successfully.

In [2]:
class Item:
    def __init__(self, name: str, price: float):
        self.name = name
        self.price = price

    def apply_discount(self, rate: float) -> None:
        self.price -= self.price * rate  

item = Item("laptop", 1000.0)
item.apply_discount(0.10)
print(item.price)    

900.0


In [18]:
class Item:
    def __init__(self, name: str, price: float):
        self.name = name
        self.price = price

    @property
    def price(self) -> float:
        return self._price

    @price.setter
    def price(self, value: float) -> None:
        if value < 0:
            raise ValueError("Price cannot be negative!")
        self._price = value

item = Item("laptop", 1000.0)
print(item.price)    

1000.0


In [25]:
class Item:
    def __init__(self, name: str, price: float, quantity: int):
        self.name = name
        self.price = price
        self.quantity = quantity

    @property
    def quantity(self) -> int:
        return self._quantity

    @quantity.setter
    def quantity(self, value: int) -> None:
        if value < 0:
            raise ValueError("Quantity cannot be negative!")
        self._quantity = value

    def apply_discount(self, rate: float) -> None:
        self.price -= self.price * rate


item = Item("Laptop", 1000.0, 5)
print(item._quantity)

5


In [ ]:
class Inventory:
    def __init__(self):
        self.items = []

    def add_item(self, item: Item) -> None:
        self.items.append(item)

    def total_value(self) -> float:
        return sum(item.price * item.quantity for item in self.items)

In [29]:
class OutOfStockError(Exception):
    """Raised when a requested stock withdrawal exceeds available quantity."""
    pass


class Item:
    def __init__(self, name: str, price: float, quantity: int):
        self.name = name
        self.price = price
        self.quantity = quantity

    @property
    def quantity(self) -> int:
        return self._quantity

    @quantity.setter
    def quantity(self, value: int) -> None:
        if value < 0:
            raise ValueError("Quantity cannot be negative!")
        self._quantity = value

    def apply_discount(self, rate: float) -> None:
        self.price -= self.price * rate

    def withdraw_stock(self, amount: int) -> None:
        if amount > self.quantity:
            raise OutOfStockError(
                f"Cannot withdraw {amount} units of {self.name}. "
                f"Only {self.quantity} units are available."
            )

        self.quantity -= amount


laptop = Item("Laptop", 1000.0, 5)

try:
    laptop.withdraw_stock(10)
except OutOfStockError as error:
    print(error)

print(f"Remaining stock: {laptop.quantity}")

Cannot withdraw 10 units of Laptop. Only 5 units are available.
Remaining stock: 5


In [30]:
from dataclasses import dataclass

@dataclass
class Product:
    sku: str
    name: str
    price: float
    category: str = "General"


product1 = Product("SKU123", "Mouse", 25.0)
product2 = Product("SKU123", "Mouse", 25.0)

print(product1)

print(product1 == product2)

Product(sku='SKU123', name='Mouse', price=25.0, category='General')
True


In [31]:
from dataclasses import dataclass


@dataclass
class Product:
    sku: str
    name: str
    price: float
    category: str = "General"

class OutOfStockError(Exception):
    pass

class InventoryItem:
    def __init__(self, product: Product, quantity: int):
        self.product = product
        self.quantity = quantity

    def withdraw_stock(self, amount: int) -> None:
        if amount > self.quantity:
            raise OutOfStockError(
                f"Cannot withdraw {amount} units of {self.product.name}. "
                f"Only {self.quantity} units are available."
            )

        self.quantity -= amount

    def total_value(self) -> float:
        return self.product.price * self.quantity

class Inventory:
    def __init__(self):
        self.items = []

    def add_item(self, item: InventoryItem) -> None:
        self.items.append(item)

    def total_value(self) -> float:
        return sum(item.total_value() for item in self.items)

product1 = Product("SKU123", "Mouse", 25.0)
product2 = Product("SKU456", "Keyboard", 75.0)

item1 = InventoryItem(product1, 10)
item2 = InventoryItem(product2, 5)

inventory = Inventory()
inventory.add_item(item1)
inventory.add_item(item2)

try:
    item1.withdraw_stock(15)
except OutOfStockError as error:
    print(f"Stock error: {error}")

print(f"Total inventory value: {inventory.total_value():.2f}")

Stock error: Cannot withdraw 15 units of Mouse. Only 10 units are available.
Total inventory value: 625.00


## Completion Note

All cells executed successfully, the local environment has passed the Day 1 technical validation.